<div align="center">

# Patra Toolkit: Model Card & Datasheet Demo

</div>

This notebook is a focused, end-to-end walkthrough of the core Patra Toolkit operations against the Postgres-backed Patra server:

1. Model Card creation
2. Datasheet creation
3. Model Card submission
4. Datasheet submission
5. Model Card listing
6. Datasheet listing
7. Model Card retrieval
8. Datasheet retrieval

It uses placeholder metadata rather than training a real model, so it can be run end-to-end quickly. For a fuller example that trains a real model and runs fairness/explainability scanners, see `GettingStarted.ipynb`.

In [ ]:
!pip install patra-toolkit

In [ ]:
from patra_toolkit import ModelCard, AIModel, Datasheet

## Configuration

Set the base URL of your Patra server. If you're running the backend locally via `docker compose -f docker-compose.backend.yml up --build` in `patra-knowledge-base`, this defaults to `http://localhost:8000`.

In [ ]:
patra_server_url = "http://localhost:8000"

## 1. Model Card Creation

Only `name` is required on `ModelCard` and `AIModel` -- every other field is optional. Attach an `AIModel` to describe the underlying model.

In [ ]:
mc = ModelCard(
    name="Demo_Classification_Model",
    version="1.0",
    short_description="A placeholder model card for demonstrating the Patra Toolkit API.",
    full_description=(
        "This model card does not describe a real trained model -- it exists purely to demonstrate "
        "model card creation, submission, listing, and retrieval."
    ),
    keywords="demo, patra, toolkit",
    author="Demo Author",
    input_type="Tabular",
    category="classification",
)

ai_model = AIModel(
    name="DemoModel",
    version="1.0",
    description="Placeholder AI model metadata.",
    owner="Demo Author",
    framework="sklearn",
    model_type="random_forest",
    test_accuracy=0.87,
)
ai_model.add_metric("Precision", 0.85)
ai_model.add_metric("Recall", 0.83)

mc.ai_model = ai_model
mc.validate()

## 2. Datasheet Creation

Datasheets describe datasets using DataCite-style metadata. No fields are required to construct one, but `validate()`/`submit()` require at least one title and one creator.

In [ ]:
ds = Datasheet(publication_year=2025, version="1.0")
ds.add_title("Demo Dataset")
ds.add_creator("Demo Author")
ds.add_description("A placeholder dataset for demonstrating the Patra Toolkit API.", "Abstract")

ds.validate()

## [Optional] TAPIS Authentication

Patra servers hosted as TAPIS pods require authentication using a JWT for secure access. To generate this token, authenticate with your TACC credentials. If you do not already have a TACC account, you can create one at [https://accounts.tacc.utexas.edu/begin](https://accounts.tacc.utexas.edu/begin). If your Patra server doesn't require authentication, skip this cell and pass `token=None` when submitting.

In [ ]:
tapis_token = mc.authenticate(username="<your_tacc_username>", password="<your_tacc_password>")
# tapis_token = None  # uncomment if your Patra server doesn't require authentication

## 3. Model Card Submission

In [ ]:
mc_result = mc.submit(patra_server_url=patra_server_url, token=tapis_token)
print(mc_result)
print("Model Card uuid:", mc.uuid)

## 4. Datasheet Submission

In [ ]:
ds_result = ds.submit(patra_server_url=patra_server_url, token=tapis_token)
print(ds_result)
print("Datasheet uuid:", ds.uuid)

## 5. Model Card Listing

Returns summaries (`uuid`, `name`, and other summary fields) for model cards on the server.

In [ ]:
ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, q="Demo_Classification_Model")

## 6. Datasheet Listing

Returns summaries (`uuid`, `title`, and other summary fields) for datasheets on the server.

In [ ]:
Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, q="Demo Dataset")

## 7. Model Card Retrieval

Retrieves the full record for a single model card by `uuid`, including its nested `ai_model` details.

In [ ]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=mc.uuid, token=tapis_token)

## 8. Datasheet Retrieval

Retrieves the full DataCite-style record for a single datasheet by `uuid`.

In [ ]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=ds.uuid, token=tapis_token)

## Summary

By following this notebook, you have:
1. Created a Model Card and a Datasheet
2. [Optionally] Authenticated with TAPIS to obtain a token
3. Submitted both the Model Card and the Datasheet to a Patra server
4. Listed model cards and datasheets on the server
5. Retrieved a single model card and a single datasheet by `uuid`

## 9. Inference Experiment + Streaming to CKN

This section runs a small real inference experiment -- get a Model Card and Datasheet from Patra, download images and a pretrained model, run inference over the images in a loop, and stream per-image inference metrics as Kafka events into CKN (`cyberinfrastructure-knowledge-network`). Those events flow through an existing Kafka Connect sink into Patra's Postgres `events` table, where they're visible in the Patra frontend's **Digital Agriculture** experiments page.

This section needs more than just the Patra backend from earlier in this notebook -- it also needs the CKN stack running and joined to the same Docker network, plus a one-time demo user registered in Postgres. See the prerequisites below.

### Prerequisites

1. **Start CKN first** (creates the `ckn-network` Docker network the Patra backend also joins):
   ```bash
   cd cyberinfrastructure-knowledge-network
   make up
   ```
2. **Then start (or restart) the Patra backend**, so its `postgres` service attaches to the now-existing `ckn-network`:
   ```bash
   cd patra-knowledge-base
   docker compose -f docker-compose.backend.yml up --build -d
   ```
   Order matters -- `ckn-network` must already exist before this command's `postgres` service can join it.
3. **Docker must be available from this notebook's environment** -- this section shells out to the `docker` CLI (build/run a small producer container, and `docker exec` into `patra-postgres`) since CKN's Kafka broker is only reachable from inside the `ckn-network` Docker network, not from the host.
4. **Important**: the CKN Postgres sink connector (`ckn_broker/pgsink-oracle-events-connector.json`) has a hardcoded target -- `jdbc:postgresql://patra-postgres:5432/patra`, i.e. the *local* Docker Postgres container -- completely independent of whatever `DATABASE_URL` `patra-backend` itself is using. If your `patra-knowledge-base` checkout has a `.env` file overriding `DATABASE_URL` to point somewhere else (e.g. a remote/shared deployment), the Model Card you submit in 9.1 and the events you stream in 9.7 will land in **two different databases**, and the event's `model_id` won't resolve. Make sure `patra_server_url` in this notebook and the CKN connector are pointed at the same Postgres for this section to work end-to-end.

In [ ]:
!pip install torch torchvision pillow

In [ ]:
import json
import os
import subprocess
import uuid as uuid_lib
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch
import torchvision
from PIL import Image
from torchvision import transforms

from patra_toolkit.datasheet import DatasheetAlternateIdentifier

### 9.1 Build and submit a Model Card + Datasheet for the inference model

These are new, separate objects from the `mc`/`ds` used earlier in this notebook -- keeping them distinct means re-running the notebook top-to-bottom stays coherent, since the earlier list/retrieval cells describe the placeholder sklearn card, not this one.

`ai_model.location` is set to torchvision's real, publicly downloadable MobileNetV2 weights URL -- reading `weights.url` here doesn't download anything; the download happens explicitly in step 9.4.

In [ ]:
weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1
weights_url = weights.url
imagenet_categories = weights.meta["categories"]
top1_acc = weights.meta["_metrics"]["ImageNet-1K"]["acc@1"] / 100.0

inference_ai_model = AIModel(
    name="MobileNetV2_ImageNet",
    version="1.0",
    description="Torchvision MobileNetV2 CNN pretrained on ImageNet-1k; used for a live inference-streaming demo.",
    owner="Demo Author",
    location=weights_url,
    license="BSD-3-Clause",
    framework="pytorch",
    model_type="cnn",
    test_accuracy=round(top1_acc, 5),
    inference_labels=imagenet_categories,
)

inference_mc = ModelCard(
    name="MobileNetV2_Inference_Demo",
    version="1.0",
    short_description="Real pretrained MobileNetV2 used to demonstrate inference + CKN streaming.",
    full_description=(
        "Downloads a real ImageNet-pretrained MobileNetV2 checkpoint via ai_model.location, runs it "
        "over ~20 real images, and streams per-image inference metrics to CKN."
    ),
    keywords="demo, patra, ckn, inference, mobilenetv2",
    author="Demo Author",
    input_type="Image",
    category="classification",
)
inference_mc.ai_model = inference_ai_model
inference_mc.validate()

inference_ds = Datasheet(publication_year=2026, version="1.0")
inference_ds.add_title("CKN Inference Demo Images")
inference_ds.add_creator("Demo Author")
inference_ds.alternate_identifiers.append(
    DatasheetAlternateIdentifier(alternate_identifier="https://picsum.photos", alternate_identifier_type="URL")
)
inference_ds.add_description(
    "20 images fetched from Lorem Picsum via https://picsum.photos/id/{0..19}/224/224 for a live "
    "inference-streaming demo. These are arbitrary real-world photographs with no ImageNet ground-truth labels.",
    "TechnicalInfo",
)
inference_ds.validate()

In [ ]:
inference_mc_result = inference_mc.submit(patra_server_url=patra_server_url, token=tapis_token)
inference_ds_result = inference_ds.submit(patra_server_url=patra_server_url, token=tapis_token)
print("Model Card uuid:", inference_mc.uuid)
print("Datasheet uuid:", inference_ds.uuid)

### 9.2 Retrieve them back from Patra

Fetches the Model Card and Datasheet back from the server by `uuid` -- this is the "get a datasheet and model card from Patra" step, rather than just reusing the in-memory objects.

In [ ]:
mc_detail = ModelCard.get_model_card(server_url=patra_server_url, uuid=inference_mc.uuid, token=tapis_token)
ds_detail = Datasheet.get_datasheet(server_url=patra_server_url, uuid=inference_ds.uuid, token=tapis_token)
model_location = mc_detail["ai_model"]["location"]
print("Model location:", model_location)
print("Datasheet titles:", ds_detail["titles"])

### 9.3 Download ~20 images

Fetches images from the URL recorded in the Datasheet.

In [ ]:
os.makedirs("ckn_demo_images", exist_ok=True)

for i in range(20):
    resp = requests.get(f"https://picsum.photos/id/{i}/224/224", stream=True, timeout=30)
    resp.raise_for_status()
    with open(f"ckn_demo_images/{i}.jpg", "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)

print("Downloaded 20 images to ./ckn_demo_images/")

### 9.4 Download and instantiate the model from `ai_model.location`

Downloads the actual weights file referenced by the retrieved Model Card, then builds the model architecture and loads the downloaded weights into it.

In [ ]:
weights_path = "mobilenet_v2_downloaded.pth"

resp = requests.get(model_location, stream=True, timeout=60)
resp.raise_for_status()
with open(weights_path, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        f.write(chunk)

model = torchvision.models.mobilenet_v2(weights=None)
model.load_state_dict(torch.load(weights_path, map_location="cpu"))
model.eval()
print(f"Loaded MobileNetV2 from {weights_path} (downloaded from {model_location})")

### 9.5 Run inference and build per-image events

Loops over the 20 images calling the model (the `model.predict()`-equivalent step for a PyTorch model), and builds one CKN event per image. The `events` table's detection-style aggregate fields (`total_images`, `true_positives`, `precision`, `mean_iou`, `map_50`, ...) come from the camera-trap/object-detection use case CKN was originally built for -- this demo is plain image classification against unlabeled photos, so those fields are populated with clearly-marked illustrative placeholders rather than a fabricated ground truth.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def build_event(image_path, running_count, top_label, top_prob, topk_scores,
                 experiment_id, user_id, device_id, model_id, domain="digital-ag"):
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    return {
        "domain": domain,
        "device_id": device_id,
        "experiment_id": experiment_id,
        "user_id": user_id,
        "model_id": model_id,  # inference_mc.uuid -- bare, no "-model" suffix needed
        "UUID": str(uuid_lib.uuid4()),  # renamed to the `uuid` column by the sink connector's transform
        "image_name": os.path.basename(image_path),
        "ground_truth": None,  # no real label available for these photos
        "image_count": running_count,
        "image_receiving_timestamp": now,
        "image_scoring_timestamp": now,
        "image_store_delete_time": now,
        "image_decision": "Save",
        "label": top_label,
        "probability": round(float(top_prob), 7),
        "flattened_scores": json.dumps(topk_scores),
        # Illustrative placeholders only -- plain image classification has no real ground truth
        # here, unlike the camera-trap/object-detection use case these columns were designed for.
        "total_images": running_count,
        "total_predictions": 1,
        "total_ground_truth_objects": 0,
        "true_positives": 0,
        "false_positives": 0,
        "false_negatives": 0,
        "precision": round(float(top_prob), 5),
        "recall": round(float(top_prob), 5),
        "f1_score": round(float(top_prob), 5),
        "mean_iou": 0.0,
        "map_50": 0.0,
        "map_50_95": 0.0,
    }


experiment_id = f"ckn-demo-{uuid_lib.uuid4().hex[:8]}"
user_id = "demo-user"
device_id = "demo-edge-device"

events = []
for i in range(20):
    image_path = f"ckn_demo_images/{i}.jpg"
    img = Image.open(image_path).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_idx = torch.topk(probs, 5)

    top_label = imagenet_categories[top5_idx[0].item()]
    top_prob = top5_prob[0].item()
    topk_scores = [
        {"label": imagenet_categories[idx.item()], "probability": round(prob.item(), 4)}
        for prob, idx in zip(top5_prob, top5_idx)
    ]

    event = build_event(
        image_path=image_path, running_count=i + 1,
        top_label=top_label, top_prob=top_prob, topk_scores=topk_scores,
        experiment_id=experiment_id, user_id=user_id, device_id=device_id,
        model_id=inference_mc.uuid,
    )
    events.append(event)
    print(f"[{i + 1}/20] {image_path}: {top_label} ({top_prob:.3f})")

print(f"Built {len(events)} events for experiment '{experiment_id}'")

### 9.6 Prepare the CKN producer container

CKN's Kafka broker (`broker:29092`) is only reachable from inside the `ckn-network` Docker network -- the host port mapping doesn't work for external clients because the broker advertises `broker:29092` as its own address, which doesn't resolve outside Docker. So instead of producing directly from this notebook's kernel, we build a tiny throwaway container (generalizing the pattern from `cyberinfrastructure-knowledge-network/examples/daemon.py`) and run it attached to `ckn-network`. This needs zero changes to CKN itself -- it reuses the existing `oracle-events` topic and its already-registered Postgres sink connector.

Two things the generated producer script does differently from `examples/daemon.py`, discovered by actually running this against a live connector: the connector config sets `value.converter.schemas.enable=true`, so bare flat JSON (what `daemon.py` itself sends) is rejected -- messages must be wrapped in a `{"schema": ..., "payload": ...}` envelope. And the three timestamp fields need Kafka Connect's logical `Timestamp` type (epoch milliseconds), not plain ISO-8601 strings, or the JDBC sink fails inserting into the `timestamptz` columns with a type-mismatch error.

In [ ]:
producer_dir = Path("ckn_producer")
producer_dir.mkdir(exist_ok=True)

(producer_dir / "produce_events.py").write_text('''\
import json, logging, os, sys, time
from datetime import datetime, timezone
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient

KAFKA_BROKER = os.getenv("CKN_KAFKA_BROKER", "broker:29092")
KAFKA_TOPIC = os.getenv("CKN_KAFKA_TOPIC", "oracle-events")
EVENTS_FILE = os.getenv("EVENTS_FILE", "/data/ckn_events.jsonl")

# The pgsink-oracle-events-connector sets value.converter.schemas.enable=true, so bare
# flat JSON (as CKN'\''s own examples/daemon.py sends) is rejected with a DataException --
# messages must be wrapped in a {"schema": ..., "payload": ...} envelope. Timestamp fields
# additionally need the Kafka Connect logical Timestamp type (epoch millis), not plain
# ISO-8601 strings, or the JDBC sink fails inserting into the timestamptz columns.
STRING_FIELDS = [
    "domain", "device_id", "experiment_id", "user_id", "model_id", "UUID",
    "image_name", "ground_truth", "image_decision", "label", "flattened_scores",
]
INT_FIELDS = [
    "image_count", "total_images", "total_predictions", "total_ground_truth_objects",
    "true_positives", "false_positives", "false_negatives",
]
FLOAT_FIELDS = ["probability", "precision", "recall", "f1_score", "mean_iou", "map_50", "map_50_95"]
TIMESTAMP_FIELDS = ["image_receiving_timestamp", "image_scoring_timestamp", "image_store_delete_time"]

CONNECT_SCHEMA = {
    "type": "struct",
    "optional": False,
    "name": "oracle_event",
    "fields": (
        [{"field": f, "type": "string", "optional": True} for f in STRING_FIELDS]
        + [{"field": f, "type": "int32", "optional": True} for f in INT_FIELDS]
        + [{"field": f, "type": "double", "optional": True} for f in FLOAT_FIELDS]
        + [
            {"field": f, "type": "int64", "optional": True,
             "name": "org.apache.kafka.connect.data.Timestamp", "version": 1}
            for f in TIMESTAMP_FIELDS
        ]
    ),
}


def _iso_to_epoch_millis(value):
    if value is None:
        return None
    dt = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)


def envelope(event):
    payload = dict(event)
    for field in TIMESTAMP_FIELDS:
        if field in payload:
            payload[field] = _iso_to_epoch_millis(payload[field])
    return {"schema": CONNECT_SCHEMA, "payload": payload}


def test_ckn_broker_connection(bootstrap_servers, timeout=10, num_tries=5):
    config = {"bootstrap.servers": bootstrap_servers}
    for i in range(num_tries):
        try:
            AdminClient(config).list_topics(timeout=timeout)
            return True
        except Exception as e:
            logging.info(f"CKN broker not available yet: {e}. Retrying in 5 seconds...")
            time.sleep(5)
    return False


def delivery_report(err, msg):
    if err is not None:
        logging.error("Delivery failed: %s", err)
    else:
        logging.info("Produced event to '%s' [partition %s]", msg.topic(), msg.partition())


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    if not test_ckn_broker_connection(KAFKA_BROKER):
        logging.error("Could not reach CKN broker at %s", KAFKA_BROKER)
        sys.exit(1)

    producer = Producer({"bootstrap.servers": KAFKA_BROKER})
    sent = 0
    with open(EVENTS_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            event = json.loads(line)
            producer.produce(KAFKA_TOPIC, json.dumps(envelope(event)), callback=delivery_report)
            producer.poll(0)
            sent += 1
    producer.flush(timeout=10)
    logging.info("Produced %d events to '%s'", sent, KAFKA_TOPIC)
''')

(producer_dir / "Dockerfile").write_text('''\
FROM python:3.11-slim
WORKDIR /app
COPY produce_events.py /app/
RUN pip install --no-cache-dir confluent-kafka
ENTRYPOINT ["python", "-u", "/app/produce_events.py"]
''')

network_check = subprocess.run(["docker", "network", "inspect", "ckn-network"], capture_output=True)
if network_check.returncode != 0:
    raise RuntimeError(
        "ckn-network not found. Run `cd cyberinfrastructure-knowledge-network && make up` first."
    )

subprocess.run(["docker", "build", "-t", "patra-ckn-producer", str(producer_dir)], check=True)
print("Producer image built.")

### 9.7 Register the demo user and stream events

The `events` table's fan-out trigger requires `user_id` to already exist in the `users` table -- there's no auto-create and no REST endpoint for it, so this registers a demo user directly (idempotent, safe to re-run). `device_id` auto-registers on first use, so it needs no setup.

In [ ]:
subprocess.run(
    ["docker", "exec", "patra-postgres", "psql", "-U", "patra", "-d", "patra",
     "-c", "INSERT INTO users (username) VALUES ('demo-user') ON CONFLICT DO NOTHING;"],
    check=True,
)
print("Demo user registered.")

events_path = producer_dir / "ckn_events.jsonl"
with open(events_path, "w") as f:
    for event in events:
        f.write(json.dumps(event) + "\n")

result = subprocess.run(
    [
        "docker", "run", "--rm", "--network", "ckn-network",
        "-v", f"{events_path.resolve()}:/data/ckn_events.jsonl:ro",
        "-e", "CKN_KAFKA_BROKER=broker:29092",
        "-e", "CKN_KAFKA_TOPIC=oracle-events",
        "patra-ckn-producer",
    ],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()
print(f"Streamed {len(events)} events to CKN for experiment '{experiment_id}'")

### 9.8 View results in the Patra frontend

1. Backend: `ENABLE_DOMAIN_EXPERIMENTS` already defaults to `true` in `docker-compose.backend.yml`.
2. Frontend: set `VITE_SUPPORTS_DOMAIN_EXPERIMENTS=true` in `patra-frontend/app/.env` (it defaults to `false`), then `npm run dev` from `patra-frontend/app/`.
3. Open the app and click **Digital Agriculture** under Experiments in the sidebar, then select `demo-user` to see this run's summary and per-image results.

You can also check the same data the frontend reads directly via the REST API:
```bash
curl -s http://localhost:8000/experiments/digital-ag/users
curl -s http://localhost:8000/experiments/digital-ag/users/demo-user/summary
curl -s http://localhost:8000/experiments/digital-ag/users/demo-user/list
```

**If nothing shows up**, check `docker logs kafka-connect` for errors around the time you streamed events -- the sink connector silently drops malformed or unresolvable records (e.g. an unregistered `user_id` or `model_id`) rather than raising anything visible here.